# Réduire la dimension : ACP et SVD

Les slides ont calculé l'ACP par SVD sur les chiffres manuscrits de scikit-learn (`load_digits` : 1 797 images 8 × 8 en niveaux de gris de 0 à 16, soit 64 colonnes). Ce notebook la pratique comme une compression des données, avant le modèle :

1. La concentration des distances
2. L'ACP par SVD
3. Choisir k
4. L'ACP dans un pipeline
5. Données creuses : `TruncatedSVD`
6. Données par lots : `IncrementalPCA`
7. Projeter de nouveaux points : ACP, UMAP, t-SNE

Les exercices sont indépendants, du plus simple au plus long : les derniers sont facultatifs. Pas besoin de GPU.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/20newsgroups/20news-bydate.tar.gz
!tar xzf 20news-bydate.tar.gz

import time

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size=0.25, stratify=y, random_state=0
)
print(X.shape, X_train.shape, X_test.shape)


def show_digits(images: np.ndarray, title: str) -> None:
  # Affiche côte à côte des images 8 × 8, une par ligne de `images`
  fig, axes = plt.subplots(1, len(images), figsize=(len(images), 1.5))
  for ax, image in zip(axes, images):
    ax.imshow(image.reshape(8, 8), cmap="gray_r")
    ax.axis("off")
  fig.suptitle(title)
  plt.show()


show_digits(X[:10], "Les 10 premières images")

## 1. La concentration des distances

En grande dimension, le plus proche et le plus lointain voisin d'un point sont presque à la même distance (la slide « La malédiction des grandes dimensions »). Vérifiez-le :

- écrivez `distance_ratio(points, query)`, le rapport entre la plus grande et la plus petite distance euclidienne de `query` aux lignes de `points` ;
- pour d = 2, 10, 100, 1 000 et 10 000, tirez 1 000 points uniformes dans $[0, 1]^d$ et un point requête, et affichez la médiane de ce rapport sur 20 tirages ;
- puis sur les chiffres : la médiane du rapport pour 20 images face aux 1 796 autres, et pour 1 796 points uniformes de dimension 64.

Les chiffres se comportent-ils comme des points uniformes de même dimension ? Qu'est-ce que cela dit du nombre de directions dont ils ont vraiment besoin ?

**Documentation utile**

- [`numpy.linalg.norm`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html)
- [`numpy.random.Generator.random`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.random.html)
- [`numpy.delete`](https://numpy.org/doc/stable/reference/generated/numpy.delete.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `np.linalg.norm(points - query, axis=1)` : la distance de `query` à chaque ligne de `points` (numpy répète `query` sur chaque ligne)
- `rng = np.random.default_rng(0)`, puis `rng.random((1_000, d))` pour 1 000 points et `rng.random(d)` pour la requête
- `np.delete(X, i, axis=0)` : toutes les images sauf la `i`-ième ; `rng.choice(len(X), 20, replace=False)` tire 20 indices distincts
- l'ordre de grandeur attendu : plus de 100 en dimension 2, environ 1,03 en dimension 10 000

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def distance_ratio(points: np.ndarray, query: np.ndarray) -> float:
  distances = ...
  return ...


rng = np.random.default_rng(0)
for d in [2, 10, 100, 1_000, 10_000]:
  ratios = [distance_ratio(..., ...) for _ in range(20)]
  print(f"d = {d} : {np.median(ratios):.2f}")

indices = rng.choice(...)
ratios = [distance_ratio(np.delete(X, i, axis=0), X[i]) for i in indices]
print(f"Chiffres (d = 64) : {np.median(ratios):.2f}")
ratios = [distance_ratio(..., ...) for _ in range(20)]
print(f"Uniformes (d = 64) : {np.median(ratios):.2f}")
```

### Solution

In [ ]:
def distance_ratio(points: np.ndarray, query: np.ndarray) -> float:
  distances = np.linalg.norm(points - query, axis=1)
  return distances.max() / distances.min()


rng = np.random.default_rng(0)
for d in [2, 10, 100, 1_000, 10_000]:
  ratios = [distance_ratio(rng.random((1_000, d)), rng.random(d)) for _ in range(20)]
  print(f"d = {d} : {np.median(ratios):.2f}")

indices = rng.choice(len(X), 20, replace=False)
ratios = [distance_ratio(np.delete(X, i, axis=0), X[i]) for i in indices]
print(f"Chiffres (d = 64) : {np.median(ratios):.2f}")
ratios = [distance_ratio(rng.random((1_796, 64)), rng.random(64)) for _ in range(20)]
print(f"Uniformes (d = 64) : {np.median(ratios):.2f}")
# Les chiffres gardent un rapport bien plus grand que des points uniformes de
# dimension 64 : ils n'occupent qu'une petite partie de l'espace, autour de bien moins
# de 64 directions. C'est ce que l'ACP exploite.

## 2. L'ACP par SVD

Calculez l'ACP de `X` comme sur la slide « ACP par SVD — Code » :

- centrez `X`, décomposez-le avec `np.linalg.svd`, puis calculez les k = 10 nouvelles colonnes `Z` et la part de variance de chaque axe ;
- vérifiez que `PCA(n_components=10)` de scikit-learn donne les mêmes colonnes, au signe près, et les mêmes parts de variance (`explained_variance_ratio_`) ;
- pour k = 2, 5, 10 et 20, reconstruisez `X` depuis ses k premières colonnes, affichez les 10 premières images reconstruites avec `show_digits`, et l'erreur quadratique moyenne de la reconstruction.

À partir de quel k reconnaît-on les chiffres ? Combien de valeurs stocke-t-on alors par image, au lieu de 64 ?

**Documentation utile**

- [`PCA.inverse_transform`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html#sklearn.decomposition.PCA.inverse_transform)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `X.mean(axis=0)` : la moyenne de chaque colonne ; `np.linalg.svd(Xc, full_matrices=False)` renvoie `U`, `S` (les valeurs singulières $\sigma_i$, déjà triées par ordre décroissant) et `Vt`
- les k nouvelles colonnes : `Xc @ Vt[:k].T` ; les parts de variance : `S**2 / np.sum(S**2)`
- comparer au signe près : `np.allclose(np.abs(a), np.abs(b))`, qui doit afficher `True`
- reconstruire : `Z @ Vt[:k] + mean`, avec les k colonnes `Z` (ou `pca.inverse_transform(Z)` avec une `PCA(k)` ajustée) ; l'erreur : `np.mean((X - X_hat) ** 2)`
- avec k = 64, la reconstruction est exacte : l'erreur vaut 0, aux arrondis près

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from sklearn.decomposition import PCA

mean = ...
Xc = X - mean
U, S, Vt = np.linalg.svd(...)
k = 10
Z = ...
ratio = ...

pca = PCA(n_components=k)
Z_sk = pca.fit_transform(X)
print(np.allclose(np.abs(Z_sk), np.abs(Z)))
print(np.allclose(..., ratio[:k]))

for k in [2, 5, 10, 20]:
  X_hat = ...
  error = np.mean((X - X_hat) ** 2)
  show_digits(X_hat[:10], f"k = {k}, erreur quadratique moyenne : {error:.2f}")
show_digits(X[:10], "Originales")
```

### Solution

In [ ]:
from sklearn.decomposition import PCA

mean = X.mean(axis=0)
Xc = X - mean
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
k = 10
Z = Xc @ Vt[:k].T  # = U[:, :k] * S[:k]
ratio = S**2 / np.sum(S**2)

pca = PCA(n_components=k)
Z_sk = pca.fit_transform(X)
print(np.allclose(np.abs(Z_sk), np.abs(Z)))
print(np.allclose(pca.explained_variance_ratio_, ratio[:k]))

for k in [2, 5, 10, 20]:
  X_hat = (Xc @ Vt[:k].T) @ Vt[:k] + mean
  error = np.mean((X - X_hat) ** 2)
  show_digits(X_hat[:10], f"k = {k}, erreur quadratique moyenne : {error:.2f}")
show_digits(X[:10], "Originales")
# Les chiffres se lisent vers k = 10 à 20 : 10 à 20 valeurs par image au lieu de 64,
# plus les k axes et la moyenne, stockés une seule fois pour tout le jeu.

## 3. Choisir k

- Ajustez une `PCA()` qui garde tous les axes, et tracez la variance expliquée cumulée en fonction de k.
- Trouvez le plus petit k qui garde 80 %, 95 % et 99 % de la variance, et vérifiez que `PCA(0.8)`, `PCA(0.95)` et `PCA(0.99)` trouvent les mêmes (leur attribut `n_components_`).

Quelle part des 64 colonnes faut-il garder pour 95 % de la variance ? Pourquoi la courbe monte-t-elle si vite au début ?

**Documentation utile**

- [`numpy.cumsum`](https://numpy.org/doc/stable/reference/generated/numpy.cumsum.html)
- [`numpy.searchsorted`](https://numpy.org/doc/stable/reference/generated/numpy.searchsorted.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `np.cumsum(pca.explained_variance_ratio_)` : la variance gardée par les k premiers axes, pour k = 1 à 64
- `np.searchsorted(cumulative, 0.95) + 1` : le plus petit k dont la variance cumulée atteint 95 % (`searchsorted` renvoie un indice, qui commence à 0)
- `plt.axhline(0.95, color="gray", linestyle="--")` repère le seuil sur le graphique
- le résultat attendu : k = 29 pour 95 %

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from sklearn.decomposition import PCA

cumulative = np.cumsum(...)
plt.plot(np.arange(1, len(cumulative) + 1), cumulative, marker=".")
plt.axhline(0.95, color="gray", linestyle="--")
plt.xlabel("k")
plt.ylabel("Variance expliquée cumulée")
plt.show()

for share in [0.8, 0.95, 0.99]:
  k = ...
  n_components = ...
  print(f"{share:.0%} : k = {k} ({k / 64:.0%} des colonnes), PCA : {n_components}")
```

### Solution

In [ ]:
from sklearn.decomposition import PCA

cumulative = np.cumsum(PCA().fit(X).explained_variance_ratio_)
plt.plot(np.arange(1, len(cumulative) + 1), cumulative, marker=".")
plt.axhline(0.95, color="gray", linestyle="--")
plt.xlabel("k")
plt.ylabel("Variance expliquée cumulée")
plt.show()

for share in [0.8, 0.95, 0.99]:
  k = np.searchsorted(cumulative, share) + 1
  n_components = PCA(share).fit(X).n_components_
  print(f"{share:.0%} : k = {k} ({k / 64:.0%} des colonnes), PCA : {n_components}")
# 95 % de la variance avec 29 axes sur 64, moins de la moitié. Les axes sont triés par
# variance décroissante, et une poignée de directions portent l'essentiel des
# variations entre les images.

## 4. L'ACP dans un pipeline

La cellule de départ a séparé `X` en `X_train` (75 %) et `X_test` (25 %). Pour k = 5, 10, 20, 30, puis sans ACP :

- ajustez sur `X_train` un `Pipeline` (`StandardScaler`, `PCA(k)`, `KNeighborsClassifier`) ;
- mesurez le temps d'ajustement, le temps de prédiction de `X_test`, et l'exactitude sur `X_test` ;
- rangez les résultats dans un `DataFrame` pandas, une ligne par k.

Quel k garde l'exactitude obtenue sans ACP ? Sur 64 colonnes, l'ACP fait-elle gagner du temps ? Pourquoi l'ajuster dans le pipeline plutôt qu'une fois pour toutes sur tout `X` ?

**Documentation utile**

- [`sklearn.pipeline.Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)
- [`sklearn.neighbors.KNeighborsClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)
- [`sklearn.metrics.accuracy_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)
- [`time.perf_counter`](https://docs.python.org/3/library/time.html#time.perf_counter)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `Pipeline([("scaler", StandardScaler()), ("pca", PCA(k)), ("model", KNeighborsClassifier())])` ; sans ACP, la même liste sans l'étape `"pca"`
- `pipeline.fit(X_train, y_train)`, puis `pipeline.predict(X_test)` et `accuracy_score(y_test, predictions)`
- `start = time.perf_counter()` avant, `time.perf_counter() - start` après : une durée en secondes
- `pd.DataFrame(rows).T`, avec `rows[k] = {"exactitude": ..., ...}` : une ligne par k
- les temps sont de l'ordre de la milliseconde, et varient d'une exécution à l'autre : relancez la cellule avant de conclure
- le résultat attendu : une exactitude d'environ 0,97 sans ACP

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

rows = {}
for k in [5, 10, 20, 30, None]:
  steps = [("scaler", StandardScaler())]
  if k is not None:
    steps.append(...)
  steps.append(("model", KNeighborsClassifier()))
  pipeline = Pipeline(steps)
  start = time.perf_counter()
  pipeline.fit(...)
  fit_time = time.perf_counter() - start
  start = time.perf_counter()
  predictions = ...
  predict_time = time.perf_counter() - start
  rows[k or "sans ACP"] = {
    "exactitude": accuracy_score(...),
    "ajustement (ms)": 1000 * fit_time,
    "prédiction (ms)": 1000 * predict_time,
  }
pd.DataFrame(rows).T.round(3)
```

### Solution

In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

rows = {}
for k in [5, 10, 20, 30, None]:
  steps = [("scaler", StandardScaler())]
  if k is not None:
    steps.append(("pca", PCA(k)))
  steps.append(("model", KNeighborsClassifier()))
  pipeline = Pipeline(steps)
  start = time.perf_counter()
  pipeline.fit(X_train, y_train)
  fit_time = time.perf_counter() - start
  start = time.perf_counter()
  predictions = pipeline.predict(X_test)
  predict_time = time.perf_counter() - start
  rows[k or "sans ACP"] = {
    "exactitude": accuracy_score(y_test, predictions),
    "ajustement (ms)": 1000 * fit_time,
    "prédiction (ms)": 1000 * predict_time,
  }
# Vers k = 20 à 30, l'exactitude rejoint celle obtenue sans ACP. Sur 64 colonnes,
# l'ACP ne fait guère gagner de temps : son propre calcul coûte autant, voire plus, que
# ce qu'elle épargne au KNN. Elle paie en grande dimension, et cela se mesure sur le
# matériel cible. Dans le pipeline, l'ACP n'apprend que de X_train, comme le
# StandardScaler : ajustée sur tout X, elle aurait vu le test, et l'évaluation serait
# optimiste.
pd.DataFrame(rows).T.round(3)

## 5. Données creuses : `TruncatedSVD`

La cellule suivante lit 11 314 messages de 20 forums Usenet (le jeu 20 Newsgroups, que la cellule de départ a téléchargé) et les représente en TF-IDF : `X_text`, une matrice creuse, avec une colonne par mot. Puis :

- combien de colonnes, et quelle part des valeurs est non nulle ? Combien d'octets la matrice occupe-t-elle en creux (ses trois tableaux `data`, `indices` et `indptr`), et combien en occuperait-elle en dense, à 8 octets par valeur ?
- réduisez-la à 100 colonnes avec `TruncatedSVD`, et mesurez le temps de calcul et la part de la variance gardée ;
- faites de même avec `PCA`, qui centre les données sans former la matrice centrée (voir les conseils pour son solveur).

Pourquoi ne pas centrer `X_text` soi-même, avec `X_text - X_text.mean(axis=0)` (ne l'exécutez pas : il y faudrait des gigaoctets de mémoire) ? Que dit la part de variance gardée par 100 axes, comparée à celle des chiffres ?

**Documentation utile**

- [`scipy.sparse.csr_matrix`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.csr_matrix.html)
- [`sklearn.decomposition.TruncatedSVD`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html)

In [ ]:
from sklearn.datasets import load_files
from sklearn.feature_extraction.text import TfidfVectorizer

news = load_files("20news-bydate-train", encoding="latin-1")
X_text = TfidfVectorizer(min_df=2).fit_transform(news.data)
print(type(X_text).__name__, X_text.shape)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `X_text.shape` ; `X_text.nnz` : le nombre de valeurs non nulles ; `X_text.data.nbytes + X_text.indices.nbytes + X_text.indptr.nbytes` : la taille en creux
- `TruncatedSVD(n_components=100, random_state=0)`, puis `fit(X_text)` ; `svd.explained_variance_ratio_.sum()` : la part de variance gardée
- `PCA(n_components=100, svd_solver="arpack", random_state=0)` : sur une matrice creuse, le solveur `"arpack"` centre implicitement, sans densifier
- compter une à deux minutes pour les deux

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from sklearn.decomposition import PCA, TruncatedSVD

n, d = X_text.shape
sparse_bytes = ...
print(f"{d} colonnes, {X_text.nnz / (n * d):.2%} de valeurs non nulles")
print(f"En creux : {sparse_bytes / 1e6:.0f} Mo, en dense : {... / 1e9:.1f} Go")

for model in [TruncatedSVD(...), PCA(...)]:
  start = time.perf_counter()
  model.fit(X_text)
  elapsed = time.perf_counter() - start
  kept = ...
  print(f"{type(model).__name__} : {elapsed:.1f} s, variance gardée {kept:.0%}")
```

### Solution

In [ ]:
from sklearn.decomposition import PCA, TruncatedSVD

n, d = X_text.shape
sparse_bytes = X_text.data.nbytes + X_text.indices.nbytes + X_text.indptr.nbytes
print(f"{d} colonnes, {X_text.nnz / (n * d):.2%} de valeurs non nulles")
print(f"En creux : {sparse_bytes / 1e6:.0f} Mo, en dense : {8 * n * d / 1e9:.1f} Go")

for model in [
  TruncatedSVD(n_components=100, random_state=0),
  PCA(n_components=100, svd_solver="arpack", random_state=0),
]:
  start = time.perf_counter()
  model.fit(X_text)
  elapsed = time.perf_counter() - start
  kept = model.explained_variance_ratio_.sum()
  print(f"{type(model).__name__} : {elapsed:.1f} s, variance gardée {kept:.0%}")
# Environ 56 000 colonnes et 0,3 % de valeurs non nulles : une vingtaine de Mo en creux,
# 5 Go en dense. Centrer soi-même rendrait la matrice dense : une colonne moins sa
# moyenne (non nulle) n'a plus de zéros. TruncatedSVD ne centre pas ; PCA avec "arpack"
# centre implicitement, plus lentement, et garde à peu près la même variance. 100 axes
# ne gardent qu'environ 13 % de la variance du texte, quand 29 en gardent 95 % pour les
# chiffres : le vocabulaire varie dans bien plus de directions. Sur du texte, on garde
# souvent 100 à 300 axes (l'analyse sémantique latente).

## 6. Données par lots : `IncrementalPCA`

Quand les données ne tiennent pas en mémoire, `IncrementalPCA` ajuste l'ACP lot par lot. Simulez-le sur les chiffres :

- découpez `X` en 10 lots, et ajustez une `IncrementalPCA` à 10 composantes avec `partial_fit`, lot après lot ;
- comparez ses parts de variance et ses axes (`components_`) à ceux d'une `PCA` à 10 composantes ajustée sur tout `X` d'un coup.

Les axes sont-ils les mêmes ? Lesquels diffèrent le plus, et pourquoi ?

**Documentation utile**

- [`IncrementalPCA.partial_fit`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.IncrementalPCA.html#sklearn.decomposition.IncrementalPCA.partial_fit)
- [`numpy.array_split`](https://numpy.org/doc/stable/reference/generated/numpy.array_split.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `np.array_split(X, 10)` : une liste de 10 lots d'environ 180 lignes ; chaque lot doit avoir au moins autant de lignes que de composantes
- `ipca = IncrementalPCA(n_components=10)`, puis `ipca.partial_fit(batch)` pour chaque lot
- les axes sont de norme 1, et définis au signe près : `np.abs(np.sum(ipca.components_ * pca.components_, axis=1))`, le cosinus entre les axes de même rang, vaut 1 pour deux axes identiques

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from sklearn.decomposition import PCA, IncrementalPCA

ipca = IncrementalPCA(n_components=10)
for batch in ...:
  ...
pca = PCA(n_components=10).fit(X)
print("Variance, IncrementalPCA :", ipca.explained_variance_ratio_.round(3))
print("Variance, PCA :", ...)
print("Cosinus entre les axes :", ...)
```

### Solution

In [ ]:
from sklearn.decomposition import PCA, IncrementalPCA

ipca = IncrementalPCA(n_components=10)
for batch in np.array_split(X, 10):
  ipca.partial_fit(batch)
pca = PCA(n_components=10).fit(X)
print("Variance, IncrementalPCA :", ipca.explained_variance_ratio_.round(3))
print("Variance, PCA :", pca.explained_variance_ratio_.round(3))
cosines = np.abs(np.sum(ipca.components_ * pca.components_, axis=1))
print("Cosinus entre les axes :", cosines.round(3))
# Les parts de variance et les premiers axes coïncident (cosinus de 1,000). Les
# derniers s'écartent un peu (cosinus vers 0,98) : leurs variances sont proches, et
# l'ajustement par lots ne les distingue qu'approximativement.

## 7. Projeter de nouveaux points : ACP, UMAP, t-SNE

En production, la réduction de dimension doit transformer des données qu'elle n'a pas vues à l'ajustement. Comparez trois projections en 2 dimensions :

- ajustez une `PCA` et un `umap.UMAP` sur `X_train`, puis transformez `X_train` et `X_test` ;
- pour chacune : le temps d'ajustement, l'exactitude d'un `KNeighborsClassifier` ajusté sur `X_train` projeté et évalué sur `X_test` projeté, et le nuage des points de test, coloré par chiffre ;
- `sklearn.manifold.TSNE` a-t-il une méthode `transform` ? Tracez à côté le nuage que donne son `fit_transform` sur tout `X`.

Laquelle sépare le mieux les chiffres ? Lesquelles peuvent servir dans un pipeline de production ?

**Documentation utile**

- [`umap.UMAP`](https://umap-learn.readthedocs.io/en/latest/api.html)
- [`sklearn.manifold.TSNE`](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html)
- [`sklearn.neighbors.KNeighborsClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)
- [`matplotlib.axes.Axes.scatter`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.scatter.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `PCA(n_components=2)` et `umap.UMAP(n_components=2, random_state=0, n_jobs=1)` : `fit(X_train)`, puis `transform(X_train)` et `transform(X_test)` ; `random_state` fixe le résultat et impose un seul cœur, que `n_jobs=1` dit sans avertissement
- `KNeighborsClassifier().fit(Z_train, y_train).score(Z_test, y_test)` : l'exactitude sur le test projeté
- `hasattr(TSNE(), "transform")` dit si la méthode existe ; `TSNE(n_components=2, random_state=0).fit_transform(X)` place tous les points d'un coup
- `ax.scatter(Z[:, 0], Z[:, 1], c=labels, cmap="tab10", s=5)` colore chaque point par son chiffre
- compter une minute ou deux : UMAP compile son code au premier appel, et t-SNE est lent

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
import umap
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import KNeighborsClassifier

print("TSNE a une méthode transform :", hasattr(TSNE(), "transform"))

_, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, reducer in zip(axes, [PCA(n_components=2), umap.UMAP(...)]):
  start = time.perf_counter()
  reducer.fit(...)
  fit_time = time.perf_counter() - start
  Z_train, Z_test = ..., ...
  accuracy = ...
  name = type(reducer).__name__
  print(f"{name} : ajustement {fit_time:.1f} s, exactitude {accuracy:.2f}")
  ax.scatter(Z_test[:, 0], Z_test[:, 1], c=y_test, cmap="tab10", s=5)
  ax.set_title(f"{name} (test)")
Z = ...
axes[2].scatter(..., c=y, cmap="tab10", s=5)
axes[2].set_title("t-SNE (tout X)")
plt.show()
```

### Solution

In [ ]:
import umap
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import KNeighborsClassifier

print("TSNE a une méthode transform :", hasattr(TSNE(), "transform"))

_, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, reducer in zip(
  axes, [PCA(n_components=2), umap.UMAP(n_components=2, random_state=0, n_jobs=1)]
):
  start = time.perf_counter()
  reducer.fit(X_train)
  fit_time = time.perf_counter() - start
  Z_train, Z_test = reducer.transform(X_train), reducer.transform(X_test)
  accuracy = KNeighborsClassifier().fit(Z_train, y_train).score(Z_test, y_test)
  name = type(reducer).__name__
  print(f"{name} : ajustement {fit_time:.1f} s, exactitude {accuracy:.2f}")
  ax.scatter(Z_test[:, 0], Z_test[:, 1], c=y_test, cmap="tab10", s=5)
  ax.set_title(f"{name} (test)")
Z = TSNE(n_components=2, random_state=0).fit_transform(X)
axes[2].scatter(Z[:, 0], Z[:, 1], c=y, cmap="tab10", s=5)
axes[2].set_title("t-SNE (tout X)")
plt.show()
# En 2 dimensions, l'ACP mélange les chiffres (exactitude vers 0,6) ; UMAP les sépare
# presque autant que les 64 colonnes (vers 0,97). t-SNE n'a pas de transform : il ne
# place que les points de son ajustement, et un nouveau point demande de tout refaire.
# L'ACP et UMAP peuvent servir dans un pipeline ; t-SNE sert à visualiser.